## Imports

First, we'll import all the libraries we need: Gradio for the UI, TensorFlow/Keras for the models, and others for image processing and connecting to Google Drive.

In [ ]:
# HERE WE ARE JUST IMPORTING THE LIBRARITY
import gradio as gr
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, RandomFlip, RandomRotation, Lambda
# HERE WE ARA IMPORTING  THE  MODEL AND THE PREPROCESSING FUNCTION
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
import numpy as np
import cv2
import os
import sys
import warnings
# FOR UNZIPPING THE FILE
import subprocess
from google.colab import drive
#JUST FOR  SURPASSING  THE MINOR TENSOR FLOW WARKIG
warnings.filterwarnings("ignore")

## Mount Drive & Unzip Data
InThis block we are connecting to Google Drive and then unzips our 'DATA.zip' file into the Colab environment so we can use the images and models.

In [ ]:

print(" Stabilizing Environment ")
print("Connecting to Google Drive")
drive.mount('/content/drive', force_remount=True)
print("Drive mounted successfully.")

print("Unzipping the DATA.zip file from Drive")
ZIP_PATH = "/content/drive/MyDrive/DATA.zip"
TARGET_DIR = "/content/"
# HERE WE ARE USING THE SUBPROCESSING  TO RUN THE UNZIP COMMAND
subprocess.run(['unzip', '-o', '-q', ZIP_PATH, '-d', TARGET_DIR])
print(" Data is unzipped and ready in /content/.")

 Stabilizing Environment 
Connecting to Google Drive
Mounted at /content/drive
Drive mounted successfully.
Unzipping the DATA.zip file from Drive
 Data is unzipped and ready in /content/.


##  Configuration
Here, we set up our main variables: the class names, labels, F1 scores (for display), and the file paths to our saved model weights in Google Drive.

In [ ]:

# FINAL CONFIGURARONT
CLASS_NAMES = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
BINARY_LABELS = ['No Tumor Detected', 'Tumor Detected']


# HERE WE ARE DISPLAYING HTE F1 SCORE IN THE UI
BINARY_F1 = 0.8877
MULTI_F1 = 0.7873
# HERE IS THE PATH TO THE SAVED MODEL
MULTI_WEIGHTS_PATH = "/content/drive/MyDrive/MODELS/resnet_final_optimized.h5"
BINARY_WEIGHTS_PATH = "/content/drive/MyDrive/MODELS/efficientnet_binary_finetuned.h5"

## , Rebuild Models
We need to build the "empty shells" of our models exactly as they were trained so we can load the weights.

### 3.1 ResNet50 (Multi-Class)
Rebuilding the 4-class ResNet50 model, including its augmentations, preprocessing, and fine-tuning setup.

In [ ]:

# HERE WEARE REBUILDING THE MODEL ARCHS

print("Building ResNet50 (Multi-Class) architecture...")
base_model_resnet = ResNet50(weights=None, include_top=False, input_shape=(224, 224, 3), name="resnet50")
base_model_resnet.trainable = True
for layer in base_model_resnet.layers[:-65]:
    layer.trainable = False


inputs_resnet = Input(shape=(224, 224, 3))
x_res = RandomFlip('horizontal')(inputs_resnet)
x_res = RandomRotation(0.1)(x_res)
x_res = tf.keras.layers.RandomZoom(0.1)(x_res)
x_res = tf.keras.layers.RandomContrast(0.1)(x_res)
x_res = Lambda(resnet_preprocess)(x_res)
x_res = base_model_resnet(x_res, training=False)
x_res = GlobalAveragePooling2D()(x_res)
x_res = Dropout(0.3)(x_res)
outputs_resnet = Dense(4, activation='softmax')(x_res)
multi_model = Model(inputs_resnet, outputs_resnet)
print("ResNet architecture built.")

Building ResNet50 (Multi-Class) architecture...
ResNet architecture built.


here we are using EfficientNetB0 (Binary) Rebuilding the 2-class (Tumor / No-Tumor) EfficientNetB0 model, including its specific layers and preprocessing.

In [ ]:

# HERE WE ARE BUILDING THE EFFICIENTNET(BINARY )MODEL
print("Building EfficientNetB0 (Binary) architecture...")
base_model_effnet = EfficientNetB0(weights=None, include_top=False, input_shape=(224, 224, 3), name="efficientnetb0")

base_model_effnet.trainable = True
for layer in base_model_effnet.layers[:-30]:
    layer.trainable = False

inputs_effnet = Input(shape=(224, 224, 3))
x_eff = RandomFlip('horizontal')(inputs_effnet)
x_eff = RandomRotation(0.1)(x_eff)
x_eff = Lambda(effnet_preprocess, name='effnet_preprocess_input')(x_eff)
x_eff = base_model_effnet(x_eff, training=False)
x_eff = GlobalAveragePooling2D()(x_eff)
x_eff = Dropout(0.3)(x_eff)
outputs_effnet = Dense(1, activation='sigmoid')(x_eff)
binary_model = Model(inputs_effnet, outputs_effnet)
print("EfficientNet architecture built.")

Building EfficientNetB0 (Binary) architecture...
EfficientNet architecture built.


## 4. Load Weights
Now we load our saved weights (.h5files) into the empty model architectures we just built. We use atry/except block to catch any errors.

In [ ]:
# HERE WE ARE  LOADING  WAIGHTS
try:
    print(f"Loading weights for Multi-Class Model")
    multi_model.load_weights(MULTI_WEIGHTS_PATH)
    print(f"Loading weights for Binary Model")
    binary_model.load_weights(BINARY_WEIGHTS_PATH)
    print(" Dual-Architecture Models loaded successfully.")
except Exception as e:
    print(f" FATAL ERROR: Failed to load weights. {e}")
    sys.exit(1)

Loading weights for Multi-Class Model
Loading weights for Binary Model
 Dual-Architecture Models loaded successfully.


## 5. Prediction Function
This is the main function Gradio will use. It takes an image, processes it (resize, RGB, batch), runs it through *both* models, and returns the formatted results for each

In [ ]:

# HERE WE ARE BUILDING THE PREDICTION FUNCTION
def predict_mri(input_image):
    """
    Runs prediction on both models after full preprocessing.
    Grad-CAM functionality has been removed for this deployment.
    """


    # HERE WE ARE VALIDATING HTE INPUT
    if input_image is None:
        return " Please upload an MRI scan.", {"No Results": 1.0}


    if len(input_image.shape) == 2: # Grayscale image
        input_image = cv2.cvtColor(input_image, cv2.COLOR_GRAY2RGB)


    # HERE WE ARE RESIZING AND CONVERTING THE TYPE
    img = cv2.resize(input_image, (224, 224)).astype(np.float32)

    # HERE WE ARE CREATIGN  THE BATCH OF 1

    img_batch = np.expand_dims(img, axis=0)


    # HERE WE ARE RUNNING THE BINARY
    bin_pred = binary_model.predict(img_batch, verbose=0)[0][0]
    binary_result = " Tumor Detected" if bin_pred >= 0.5 else " No Tumor Detected"
    binary_output_str = f"{binary_result} (Confidence: {bin_pred * 100:.2f}%)"

    # HERE WE ARE RUNNIN THE MULTI CLASS
    multi_pred = multi_model.predict(img_batch, verbose=0)[0]
    multi_probs = {CLASS_NAMES[i]: float(multi_pred[i]) for i in range(4)}

    # HERE WE ARE  RETURNING THE FORMATTED OUTPUTS

    return binary_output_str, multi_probs

## 6. Launch Gradio UI
This final block builds the user interface. We set a clean theme, create the layout with HTML titles, add the input/output boxes, and set up the example images. Finally, we .launch() the app.

In [ ]:
# LAUNCHING
print("\n Launching Final UI")

# HERE IS SIMPLE PATH TO THE  EXAMPLE IMAGES

example_path = "/content/DATA/Testing(10%)"
examples = [
    os.path.join(example_path, "glioma_tumor/image(1).jpg"),
    os.path.join(example_path, "meningioma_tumor/image(1).jpg"),
    os.path.join(example_path, "pituitary_tumor/image(1).jpg"),
    os.path.join(example_path, "no_tumor/image(1).jpg")
]

# HERE WE ARE JUST USING THE SIMPLE THEMES  FOR THE CLARITY
with gr.Blocks(theme=gr.themes.Soft(primary_hue=gr.themes.colors.slate, secondary_hue=gr.themes.colors.gray)) as iface:
    # SIMPLE CUSTOM HTML
    gr.HTML(
        """
        <div style="text-align:center; font-size:42px; font-weight:700;">
             AI-Powered Brain Tumor Diagnostic System
        </div>
        <div style="text-align:center; font-size:20px; color:#6B7280;">
            AIRE410 • Dual-Architecture (EfficientNet + ResNet50)
        </div>
        <hr style="opacity:0.35; margin-bottom:18px;">
        """
    )

    with gr.Row(variant="panel"):
        # HERE IS OUT INPUT COLUMN
        with gr.Column(scale=1):
            img_input = gr.Image(label=" Upload MRI Scan", type="numpy", height=300)
            btn_submit = gr.Button("Run Diagnosis", variant="primary", scale=2)

        # HERE IS OUR OUTPUT CULOMN
        with gr.Column(scale=2):
            output_screening = gr.Textbox(
                label=f"Stage 1 — EfficientNet Screening (F1 = {BINARY_F1:.4f})",
                interactive=False
            )
            output_diagnosis = gr.Label(
                label=f"Stage 2 — ResNet50 Classification (F1 = {MULTI_F1:.4f})",
                num_top_classes=4
            )


    gr.Examples(
        examples=examples,
        inputs=img_input,
        outputs=[output_screening, output_diagnosis],
        fn=predict_mri,
        cache_examples=False
    )


    btn_submit.click(
        fn=predict_mri,
        inputs=img_input,
        outputs=[output_screening, output_diagnosis]
    )

# HERE WE ARE JUST LAUNCHING THE APPLICATION
iface.launch(share=True)


 Launching Final UI
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://862fc4e700c6ad269c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
